In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline

from mlops_playground.features import make_daily_orders, make_daily_raw_features, make_features
from mlops_playground.pipeline import FeatureSelector

In [ ]:
root = Path("..") if Path.cwd().name == "notebooks" else Path(".")
data = pd.read_parquet(root / "data/processed/train.parquet")
orders = make_daily_orders(data)
raw_features = make_daily_raw_features(data)
X = make_features(orders, raw_features).drop(columns="orders").dropna(axis=1, how="all")
y = orders.reindex(X.index)
feature_names = list(X.columns)

In [ ]:
pipeline = Pipeline([
    ("preprocessing", FeatureSelector(feature_names)),
    ("model", HistGradientBoostingRegressor(
        loss="poisson",
        learning_rate=0.03,
        max_iter=500,
        max_leaf_nodes=15,
        min_samples_leaf=10,
        l2_regularization=5,
        random_state=42,
    )),
])
pipeline.fit(X, y)

In [ ]:
bundle = {
    "pipeline": pipeline,
    "metadata": {
        "model_version": "1.0.0",
        "features": feature_names,
        "threshold": None,
    },
}
(root / "artifact").mkdir(exist_ok=True)
joblib.dump(bundle, root / "artifact/model.joblib")